# Bronze validation
Chạy sau hai Copy Activity. Gắn default Lakehouse. Không biến đổi Bronze.

In [ ]:
run_id = 'manual'
source_folder = 'Source'
bronze_folder = 'Bronze'
reports_folder = 'Reports'
table_schema = 'dbo'
strict_project_snapshot = True

In [ ]:
from pyspark.sql import functions as F, types as T
spark.conf.set("spark.sql.session.timeZone", "UTC")
def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
def key(*cols):
    return F.sha2(F.to_json(F.struct(*[F.col(c) for c in cols])), 256)

In [ ]:
import re
from pyspark.sql import functions as F, types as T
assert re.fullmatch(r'[A-Za-z0-9_-]+', run_id), 'Invalid run_id'
assert not table_schema or re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',table_schema)
def table_name(name):
    assert re.fullmatch(r'[a-z_]+',name)
    return f'{table_schema}.{name}' if table_schema else name
def table(name): return spark.table(table_name(name))
def save(df, name):
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(table_name(name))
def bronze_path(kind): return f'Files/{bronze_folder}/{run_id}/{kind}/'
def source_frame(filename):
    return (spark.read.option('header',True).option('inferSchema',False).option('quote','"')
        .option('escape','"').option('mode','FAILFAST').csv(f'Files/{source_folder}/{filename}'))

In [ ]:
sales = spark.read.parquet(bronze_path('sales'))
rates = spark.read.parquet(bronze_path('exchange_rate'))
expected_sales = {'order_id','order_date','customer_info','customer_age','location','device_type','items','payment_method','currency','discount_code','shipping_cost','total_amount','order_status','loyalty_points','feedback_score'}
assert set(sales.columns) == expected_sales
assert set(rates.columns) == {'year','month','from_currency','to_currency','exchange_rate'}
assert all(isinstance(f.dataType, T.StringType) for f in sales.schema.fields + rates.schema.fields), 'Copy mapping must preserve source as strings'
assert not strict_project_snapshot or sales.count() == 5250, 'Expected original project snapshot: 5250 rows'
assert rates.count() == 24
assert sales.filter(F.col('total_amount').startswith('$')).count() > 0
assert sales.filter(F.col('customer_age') == 'twenty').count() > 0
assert sales.filter(F.from_json('customer_info', 'name STRING, email STRING, phone STRING').isNull()).count() == 0
print('Bronze validated: original snapshot preserved')

In [ ]:
# Compare the entire snapshot, including duplicate multiplicities and JSON strings.
for actual, filename in [(sales,'mindx_raw_sales_data.csv'),(rates,'exchange_rate_2425.csv')]:
    original = source_frame(filename).select(*actual.columns)
    assert original.count() == actual.count(), 'Copy row count mismatch'
    assert original.exceptAll(actual).limit(1).count()==0, 'Bronze changed source values'
    assert actual.exceptAll(original).limit(1).count()==0, 'Unexpected Bronze values'
print('Bronze COPY verified against both source CSV files')